**Resources:**
- [Kalshi API Docs](https://docs.kalshi.com/welcome)
- [Python SDK (kalshi-python-sync)](https://pypi.org/project/kalshi-python-sync/)
- [OpenAPI Spec](https://docs.kalshi.com/openapi.yaml)
- [Demo Environment](https://demo.kalshi.co)

**Python 3.13** is required, the Kalshi SDK (`kalshi-python-sync`) targets the latest Python.

## 1. Setup

In [ ]:
# pip install kalshi-python-sync requests python-dotenv

In [1]:
import os
import requests
from dotenv import load_dotenv
from kalshi_python_sync import Configuration, KalshiClient

# The URL contains "elections" for historical reasons — Kalshi started as an elections market.
# This is the correct production URL and serves all market categories.
BASE_URL = "https://api.elections.kalshi.com/trade-api/v2"

# Demo environment (mock funds, separate credentials)
# BASE_URL = "https://demo-api.kalshi.co/trade-api/v2"

## 2. Exploring Events

In [30]:
response = requests.get(
    f"{BASE_URL}/events",
    params={
        "limit": 20,
        "status": "open",
        "with_nested_markets": True
    }
)
events = response.json().get("events", [])
print(f"Found {len(events)} events")

Found 20 events


In [31]:
for event in events[:5]:
    print(f"Event: {event['title']}")
    print(f"  Category: {event.get('category', 'N/A')}")
    print(f"  Series: {event.get('series_ticker', 'N/A')}")
    print(f"  Markets inside: {len(event.get('markets', []))}")
    print()

Event: Will Elon Musk visit Mars in his lifetime?
  Category: World
  Series: KXELONMARS
  Markets inside: 1

Event: Who will the next Pope be?
  Category: Elections
  Series: KXNEWPOPE
  Markets inside: 7

Event: Will the world pass 2 degrees Celsius over pre-industrial levels before 2050?
  Category: Climate and Weather
  Series: KXWARMING
  Markets inside: 1

Event: Will a human land on Mars before California starts high-speed rail?
  Category: Science and Technology
  Series: KXMARSVRAIL
  Markets inside: 1

Event: Will a supervolcano erupt before 2050?
  Category: Climate and Weather
  Series: KXERUPTSUPER
  Markets inside: 1



## 3. Markets Inside an Event

In [38]:
event = events[0]
markets = event.get("markets", [])

print(f"Event: {event['title']}")
print(f"Category: {event.get('category', 'N/A')}")
print(f"Number of markets: {len(markets)}\n")

for m in markets:
    print(f"  {m.get('yes_sub_title', m['ticker'])}")
    print(f"    Ticker: {m['ticker']}")
    print(f"    Bid: {m.get('yes_bid_dollars', 'N/A')} / Ask: {m.get('yes_ask_dollars', 'N/A')}")
    print()

Event: Will Elon Musk visit Mars in his lifetime?
Category: World
Number of markets: 1

  Mars
    Ticker: KXELONMARS-99
    Bid: 0.0800 / Ask: 0.1000



## 4. Browsing Markets Directly

In [5]:
response = requests.get(
    f"{BASE_URL}/markets",
    params={
        "limit": 20,
        "status": "open",
        # "mve_filter": "exclude",
    }
)
markets_list = response.json().get("markets", [])

for m in markets_list[:5]:
    print(f"Market: {m.get('yes_sub_title', m['ticker'])}")
    print(f"  Ticker: {m['ticker']}")
    print(f"  Event: {m.get('event_ticker', 'N/A')}")
    print(f"  Bid: {m.get('yes_bid_dollars', 'N/A')} / Ask: {m.get('yes_ask_dollars', 'N/A')}")
    print(f"  Volume 24h: {m.get('volume_24h_fp', 'N/A')}")
    print()

Market: yes West Ham,yes Real Madrid,yes Marseille,yes Milwaukee,yes Atlanta,yes San Antonio,yes Charlotte,yes Utah,yes Miami,yes Houston,yes Boston,yes Denver,yes Orlando,yes Philadelphia,yes New York,yes Roma
  Ticker: KXMVECROSSCATEGORY-S2026F7124B4EC89-00E95B71CF8
  Event: KXMVECROSSCATEGORY-S2026F7124B4EC89
  Bid: 0.0000 / Ask: 0.0000
  Volume 24h: 0.00

Market: yes Real Madrid,yes Milwaukee,yes Atlanta,yes San Antonio,yes Golden State,yes Miami,yes Houston,yes Boston,yes Denver,yes Orlando,yes Philadelphia,yes New York,yes Roma
  Ticker: KXMVESPORTSMULTIGAMEEXTENDED-S202634FB5C3EEAB-36A1C283CD7
  Event: KXMVESPORTSMULTIGAMEEXTENDED-S202634FB5C3EEAB
  Bid: 0.0000 / Ask: 0.0000
  Volume 24h: 0.00

Market: yes Atlanta,yes San Antonio,yes Charlotte,yes Houston,yes Boston,yes Orlando,yes Philadelphia,yes New York
  Ticker: KXMVECROSSCATEGORY-S2026C4E7816E496-9B58B2E1519
  Event: KXMVECROSSCATEGORY-S2026C4E7816E496
  Bid: 0.0000 / Ask: 0.0000
  Volume 24h: 0.00

Market: yes Atlanta,yes

## 5. Finding a Market

**series → events → markets**

In [10]:
# Step 1: Fetch all series, see what categories exist
resp = requests.get(f"{BASE_URL}/series")
all_series = resp.json().get("series", [])

categories = {}
for s in all_series:
    cat = s.get("category", "")
    if cat:
        categories.setdefault(cat, []).append(s)

print(f"Total series: {len(all_series)}")
print(f"\nCategories:")
for cat, items in sorted(categories.items()):
    print(f"  {cat} ({len(items)} series)")

Total series: 9549

Categories:
  Climate and Weather (264 series)
  Companies (344 series)
  Crypto (225 series)
  Economics (497 series)
  Education (2 series)
  Elections (1244 series)
  Entertainment (2303 series)
  Exotics (10 series)
  Financials (221 series)
  Health (111 series)
  Mentions (327 series)
  Politics (1812 series)
  Science and Technology (203 series)
  Social (71 series)
  Sports (1690 series)
  Transportation (40 series)
  World (142 series)


In [11]:
# Step 2: Pick a category, see what series are in it
crypto_series = categories["Crypto"]
print(f"Crypto series: {len(crypto_series)}\n")

for s in crypto_series[:10]:
    print(f"  {s['ticker']:30s} | {s['title']}")

Crypto series: 225

  KXXRPMINMON                    | XRP Min monthly
  KXTOKENLAUNCHUNIT              | Unit token
  KXNEWCOINLAUNCH                | New coin launch
  KXBTCMAX125                    | When will bitcoin hit 125k?
  ETHD                           | Ethereum price Above/below
  KXEIP3540                      | 3540
  KXAIRDROPHYPE                  | HYPE airdrop
  KXCRYPTOPERFORMY               | Crypto performance
  KXSUIATH26                     | SUI ATH 2026? 
  KXBTC15M                       | Bitcoin price up down


In [12]:
# Step 3: Pick a series, fetch its open events
resp = requests.get(
    f"{BASE_URL}/events",
    params={"series_ticker": "KXBTC15M", "status": "open", "with_nested_markets": True}
)
btc_events = resp.json().get("events", [])
print(f"Open events: {len(btc_events)}\n")

for e in btc_events:
    markets = e.get("markets", [])
    print(f"Event: {e['title']}")
    print(f"  Ticker: {e['event_ticker']}")
    print(f"  Markets: {len(markets)}")
    for m in markets:
        print(f"    {m.get('yes_sub_title', m['ticker'])} — Bid: {m.get('yes_bid_dollars', 'N/A')} / Ask: {m.get('yes_ask_dollars', 'N/A')}")
    print()

Open events: 1

Event: BTC 15 min · $71,759.89 target
  Ticker: KXBTC15M-26APR100545
  Markets: 1
    Target Price: $71,759.89 — Bid: 0.8300 / Ask: 0.8400



In [13]:
# Step 4: Pick the market
event = btc_events[0]
markets = event.get("markets", [])
ticker = markets[0]["ticker"]

print(f"Event: {event['title']}")
print(f"Market: {markets[0].get('yes_sub_title', ticker)}")
print(f"Ticker: {ticker}")
print(f"Bid: {markets[0].get('yes_bid_dollars', 'N/A')} / Ask: {markets[0].get('yes_ask_dollars', 'N/A')}")

Event: BTC 15 min · $71,759.89 target
Market: Target Price: $71,759.89
Ticker: KXBTC15M-26APR100545-45
Bid: 0.8300 / Ask: 0.8400


## 6. Single Market Deep Dive

In [14]:
response = requests.get(f"{BASE_URL}/markets/{ticker}")
market_detail = response.json().get("market", {})

# The market only has event_ticker — fetch the event to get the question
event_ticker = market_detail.get("event_ticker", "")
ev_resp = requests.get(f"{BASE_URL}/events/{event_ticker}")
event_title = ev_resp.json().get("event", {}).get("title", "N/A")

print(f"Event: {event_title}")
print(f"Market: {market_detail.get('yes_sub_title', 'N/A')}")
print(f"Ticker: {market_detail['ticker']}")
print(f"Status: {market_detail.get('status', 'N/A')}")
print(f"Type: {market_detail.get('market_type', 'N/A')}")
print(f"\nPricing:")
print(f"  Bid: ${market_detail.get('yes_bid_dollars', 'N/A')}")
print(f"  Ask: ${market_detail.get('yes_ask_dollars', 'N/A')}")
print(f"  Last price: ${market_detail.get('last_price_dollars', 'N/A')}")
print(f"\nVolume:")
print(f"  Total: {market_detail.get('volume_fp', 'N/A')} contracts")
print(f"  24h: {market_detail.get('volume_24h_fp', 'N/A')} contracts")
print(f"  Open interest: {market_detail.get('open_interest_fp', 'N/A')}")
print(f"\nSchedule:")
print(f"  Open: {market_detail.get('open_time', 'N/A')}")
print(f"  Close: {market_detail.get('close_time', 'N/A')}")

Event: BTC 15 min · $71,759.89 target
Market: Target Price: $71,759.89
Ticker: KXBTC15M-26APR100545-45
Status: active
Type: binary

Pricing:
  Bid: $0.9590
  Ask: $0.9690
  Last price: $0.9550

Volume:
  Total: 72809.69 contracts
  24h: 9474.15 contracts
  Open interest: 29636.81

Schedule:
  Open: 2026-04-10T09:30:00Z
  Close: 2026-04-10T09:45:00Z


## 7. Order Book

In [46]:
response = requests.get(
    f"{BASE_URL}/markets/{ticker}/orderbook",
    params={"depth": 10}
)
orderbook = response.json().get("orderbook_fp", {})

yes_levels = orderbook.get("yes_dollars", [])
no_levels = orderbook.get("no_dollars", [])

print(f"=== Order Book: {ticker} ===")
print(f"\nYES side ({len(yes_levels)} levels):")
for price, size in yes_levels[:5]:
    print(f"  ${price} | {size} contracts")

print(f"\nNO side ({len(no_levels)} levels):")
for price, size in no_levels[:5]:
    print(f"  ${price} | {size} contracts")

=== Order Book: KXMVESPORTSMULTIGAMEEXTENDED-S2026BC1D989725A-3DD8B7D9A09 ===

YES side (0 levels):

NO side (0 levels):


In [47]:
if yes_levels and no_levels:
    best_yes_bid = float(yes_levels[0][0])
    best_no_bid = float(no_levels[0][0])
    best_yes_ask = 1.0 - best_no_bid
    spread = best_yes_ask - best_yes_bid

    print(f"Best YES bid: ${best_yes_bid:.4f}")
    print(f"Best YES ask: ${best_yes_ask:.4f}")
    print(f"Spread: ${spread:.4f}")
    print(f"Midpoint: ${(best_yes_bid + best_yes_ask) / 2:.4f}")

In [48]:
response = requests.get(
    f"{BASE_URL}/markets/trades",
    params={"ticker": ticker, "limit": 5}
)
trades = response.json().get("trades", [])

print(f"Recent trades for {ticker}:\n")
for t in trades:
    print(f"  Price: ${t.get('yes_price_dollars', t.get('yes_price', 'N/A'))} | "
          f"Count: {t.get('count_fp', t.get('count', 'N/A'))} | "
          f"Taker: {t.get('taker_side', 'N/A')} | "
          f"Time: {t.get('created_time', 'N/A')}")

Recent trades for KXMVESPORTSMULTIGAMEEXTENDED-S2026BC1D989725A-3DD8B7D9A09:



## 8. Authentication

In [ ]:
load_dotenv()

API_KEY_ID = os.getenv("KALSHI_API_KEY_ID")
PRIVATE_KEY_PATH = os.getenv("KALSHI_PRIVATE_KEY_PATH")

config = Configuration(host=BASE_URL)

with open(PRIVATE_KEY_PATH, "r") as f:
    config.private_key_pem = f.read()

config.api_key_id = API_KEY_ID

client = KalshiClient(config)

In [ ]:
balance = client.get_balance()
print(f"Balance: ${balance.balance / 100:.2f}")

## 9. Place a Market Order

Kalshi technically only supports limit orders, but the `time_in_force` parameter changes behaviour:
- **`fill_or_kill`** — fills entirely right now, or cancels the whole order
- **`immediate_or_cancel`** — fills what it can right now, cancels the rest
- **`good_till_canceled`** — rests in the book until filled or canceled

To simulate a market order: use `fill_or_kill` and set `yes_price` to 99 (= $0.99). This is a **ceiling**, not the price you pay. Kalshi's matching engine fills at the best available price(s) in the book.

In [ ]:
bet_amount = 1.00

current_ask = float(market_detail.get('yes_ask_dollars', '0'))
count = int(bet_amount / current_ask)

print(f"Current ask: ${current_ask}")
print(f"Bet amount: ${bet_amount}")
print(f"Contracts: {count} (at ${current_ask} each = ${count * current_ask:.2f})")

result = client.create_order(
    ticker=ticker,
    action="buy",
    side="yes",
    count=count,
    yes_price=99,
    type="limit",
    time_in_force="fill_or_kill",
)
print(f"\nOrder executed!")
print(f"  Order ID: {result.order.order_id}")
print(f"  Status: {result.order.status}")

## 10. Get Open Positions

In [ ]:
positions = client.get_positions()
market_positions = positions.market_positions or []
print(f"Open positions: {len(market_positions)}\n")

for pos in market_positions:
    print(f"  Market: {pos.ticker}")
    print(f"  Position: {pos.position_fp} contracts")
    print(f"  Market exposure: ${pos.market_exposure_dollars}")
    print()

## 11. Place a Limit Order

In [ ]:
bet_amount = 1.00
price_per_contract = 0.30

count = int(bet_amount / price_per_contract)
price_cents = int(price_per_contract * 100)

print(f"Bet amount: ${bet_amount}")
print(f"Price per contract: ${price_per_contract}")
print(f"Contracts: {count}")
print(f"Total cost if filled: ${count * price_per_contract:.2f}")

result = client.create_order(
    ticker=ticker,
    action="buy",
    side="yes",
    count=count,
    yes_price=price_cents,
    type="limit",
    time_in_force="good_till_canceled",
)
limit_order_id = result.order.order_id

print(f"\nLimit order placed!")
print(f"  Order ID: {limit_order_id}")
print(f"  Status: {result.order.status}")

## 12. Find Open Orders

In [ ]:
orders_resp = client.get_orders(status="resting")
orders = orders_resp.orders or []
print(f"Resting orders: {len(orders)}\n")

for o in orders:
    print(f"  ID: {o.order_id}")
    print(f"  Market: {o.ticker}")
    print(f"  {o.action.upper()} {o.side.upper()} @ ${o.yes_price_dollars}")
    print(f"  Remaining: {o.remaining_count_fp} contracts")
    print()

## 13. Cancel Order

In [ ]:
result = client.cancel_order(order_id=limit_order_id)
print(f"Cancelled order: {limit_order_id}")